### C3M10: Data Cleaning Pt. 2

## CASE STUDY for today

Several years ago, on the first day of class, I asked students to fill out a survey.  

Here are the questions I asked:

* What is the probability you'll actually take this class  (should be a number between zero and one, inclusive)?
* What is your student status? (Undergradute, Graduate, Something Else)
* If you're a student, what year are you? (1 = freshman OR first year grad, etc.)
* In which division is your major (or likely major)? (Social Sciences, Humanities, Life Sciences, etc.)
* With which gender identiy to you most identify? (Female, Male, Gender non-conforming, Trans-gender, Choose not to answer)
* What is your height in cm?
* What is your handedness (Right handed, Left handed, Ambidextrous)
* How much (including tip) did you spend on your last haircut?
* How many minutes per week do you estimate that you exercise?
* How many minutes per day do you estimate you spend on coursework (not including time in class)?
* How many minutes per day do you estimate you surf the web or social media?   Don't include watching shows, do include watching mindless YouTube videos, time on TikTok or Instagram or Facebook or .....  Don't include anything related to academic activities.
* How many minutes per day do you estimate you watch shows?   Include Netflix, Hulu, actual TV, etc.
* On a scale of 1 to 7, how do you rate your SOCIAL political views? (1 = Extremely Conservative, 7 = Extremely Liberal, 4 = Moderate)
* On a scale of 1 to 7, how do you rate your ECONOMIC political views? (1 = Extremely Conservative, 7 = Extremely Liberal, 4 = Moderate)
* If you could be any animal, what would you be?
* How many close friends do you feel you have?
* Record your estimated pulse over 60 seconds


In [ ]:
#This is to set global options so we can actually see all columns of a dataset when we ask for them
options(repr.matrix.max.cols = 150, repr.matrix.max.rows = 200)
library(car)
library(ggplot2)
library(dplyr)


In [ ]:
#Get the survey data
survey <- read.csv("https://raw.githubusercontent.com/jreuning/sds230_data/refs/heads/main/class.survey.csv", header = TRUE, as.is = F)

#Get the dimension of the survey
dim(survey)

#What are the variables on the dataset?
names(survey)

#What data types were created?
str(survey)

How many complete survey responses do we have? `complete.cases()` creates a **LOGICAL** vector that tells us which cases are complete. 

In [ ]:
complete.cases(survey)

#This counts TRUE as 1 and FALSE as 0.
sum(complete.cases(survey))

To make a dataframe that **ONLY** has the complete cases we can do either of the following:

In [ ]:
survey_com <- survey[complete.cases(survey), ]   #only keep rows where complete.cases = TRUE
dim(survey_com)
survey_com <- na.omit(survey)  #omit rows where there are any NAs i.e. any missing values

We could create an object that ONLY has rows with missing data to see what tends to be missing.


In [ ]:
# ! is the 'not' operator.
survey_miss <- survey[!complete.cases(survey), ]
survey_miss

A few things to note:
*   Very little missing data. Five missing pulses, several missing haircuts, a few random other variables
*   The row labels on survey_miss are the row numbers according the numbering in the COMPLETE data set.

### Investigate Variables

#### Gender

Let's start by looking at the distribution of gender responses.

In [ ]:
table(survey$Gender)
levels(survey$Gender)

Since have insufficient data to say anything about levels that aren't 'Male' or 'Female' separately, I create a category that is simply 'Other or NS'

In [ ]:
survey$Gender[survey$Gender %in% levels(survey$Gender)[c(1, 3, 5)]] <- "Choose not to answer"
table(survey$Gender)
levels(survey$Gender)


We still have levels of `Gender` that we're not using, AND label is not what we want.
To fix this use the `droplevels()` function.

In [ ]:
survey$Gender <- droplevels(survey$Gender)
table(survey$Gender)

#Fix label for first level
levels(survey$Gender)[1] <- "Other or NS"

table(survey$Gender)

#### Height

We start with an overall boxplot of Height.

In [ ]:
boxplot(survey$HtCm, main = "Boxplot of Ht in cm", col = 'orange', lwd = 2)

There are some surprising values here . . .   Let's look at data.

In [ ]:
sort(survey$HtCm)

#Same thing to nearest 10th foot
round(sort(survey$HtCm)/2.5/12, 1)

Let's assume these are unknown errors (it's unlikely anyone is less than 1 foot tall).  Replace unusual values with NA (not available), i.e. Missing, i.e. **WE DON'T KNOW.**

In [ ]:
#We didn't have any super tall people, but I have had this happen in the past . .
survey$HtCm[survey$HtCm < 100 | survey$HtCm > 300] <- NA
survey$HtCm

Let's make an updated boxplot with unusual points removed.

In [ ]:
boxplot(survey$HtCm, main = "Boxplot of Ht in cm", col = 'orange', lwd = 2)

By the way, heights are usually approximately normally distributed.   Let's see if this is true by making a normal quantile plot.

In [ ]:
#Make a normal-quantile plot using qqp() in the car package
qqp(survey$HtCm, col = "red", col.lines = "blue", pch = 19, main = "N. Quantile Plot for Height (cm)")

*Next* - we make a boxplot of heights by identified gender.

In [ ]:
#The mar option is used to make more space on the left side of the graph
par(mar = c(3, 7, 2, 1))
boxplot(survey$HtCm ~ survey$Gender, main = "Boxplot of Ht in cm", col = c(2:4), 
        lwd = 2, 
        horizontal = TRUE, 
        las = 2, 
        ylab = "")

Unsurprisingly, in general people who identify as Male tend to be somewhat taller.  People identifying as Other or NS are right between Male and Female.

#### Pulse

Start with histogram of pulse overall.

In [ ]:
hist(survey$Pulse, 
     col = "orange", 
     breaks = 10, 
     main = "Histogram of Pulse",
     xlab = "Beats per minute")

Again, a couple of unusual values

In [ ]:
sort(survey$Pulse)

#I am curious who is barely alive
boxplot(Pulse ~ Gender, data = survey, main = "Boxplot of Pulse By Identified Gender", col = c(7,2,4), lwd = 2)

The American Heart Association says well trained athlete can have resting pulse of about 40.   We remove anything below 40.


In [ ]:
#We remove anything below 40.
survey$Pulse[survey$Pulse < 40] <- NA

Incidentally, do pulse values seem normally distributed?

In [ ]:
qqp(survey$Pulse, pch = 19)    
#overall, pretty close

#### Social and Economic Attitudes

First, let's look at the values of Social and Econ.

In [ ]:
#I want to see the unique values
levels(survey$Social)
levels(survey$Econ)


These are **NOT** numeric values - we need to pull off the numeric value which is always the first charachter in the string.

We use the  `substr()` function to get just the numeric values.

In [ ]:
#Example of substr()

substr("Hi Mom", 2, 4)

In [ ]:
substr(survey$Social, 1, 1)

survey$Social <- as.numeric(substr(survey$Social, 1, 1))
survey$Econ <- as.numeric(substr(survey$Econ, 1, 1))
#Make a table to see how often each pair of values occurs
table(survey$Econ, survey$Social)

Let's make a scatterplot of these two variables.

In [ ]:
#Raw Scatterplot
plot(survey$Social, survey$Econ, pch = 19, col = "red", xlab = "Social Score",
     ylab = "Economic Score")
mtext(paste("Sample Correlation =", round(cor(survey$Social, survey$Econ, use = "complete.obs"), 3)), cex = 1.2, line = 0)
mtext("Social Score vs. Economic Score", cex = 1.2, line = 1)

This is hard to read (since we have many dots on top of each other).

##### Jittering

Jittering adds a small amount of random normally distributed noise to both the x and the y values.  This has the effect of 'smearing' data around a bit so that we can see observations more clearly.

You can jitter using the `jitter()` function.

In [ ]:
#Jittered
plot(jitter(survey$Social), jitter(survey$Econ), pch = 19, col = "red", xlab = "Social Score",
     ylab = "Economic Score")
mtext(paste("Sample Correlation =", round(cor(survey$Social, survey$Econ, use = "complete.obs"), 3)), cex = 1.2, line = 0)
mtext("Social Score vs. Economic Score", cex = 1.2, line = 1)

Another solution is to make the plot character area proportional to the number of individuals at each location.

In [ ]:

#Radii Proportional plot
#Get frequencies
freq <- c(table(survey$Social, survey$Econ))
#Get corresponding vectors, call them x1 and y1
x1 <- rep(c(1:7), 7)
y1 <- sort(rep(c(1:7),7))

#Make plot - area of character is proportional to frequency of occurence (i.e. square root of count)
plot(x1, y1, pch = 19, col = "red", xlab = "Social Score",
     ylab = "Economic Score", cex = sqrt(freq), ylim = c(1, 7.5), xlim = c(1, 7.5))

mtext(paste("Sample Correlation =", round(cor(survey$Social, survey$Econ, use = "complete.obs"), 3)), cex = 1.2, line = 0)
mtext("Social Score vs. Economic Score", cex = 1.2, line = 1)
text(x1, y1, freq, cex = log(freq/max(freq)*3 + 1)+.01, col = "black")


#### What Animal Would You Be?

In the survey, I restricted the responses students could enter.  This is good data collection practice - try to limit possibilities for respondents when possible.   

For example, if I simply asked "What is your height?" and let students determine how to respond, we might get

*  5'5
*  5' 5"
*  5 feet 5 inches
*  Five feet five inches
*  65 inches
*  65"
*  162.5 cm
*  etc.

Sometimes free responses are desired - and then these need cleaning!

Let's look at `Animal` - what animal would you like to be?

In [ ]:
survey$Animal

In [ ]:
#Make a barplot
barplot(table(survey$Animal))  #not very helpful

In [ ]:
#Most common to least common - use the 'sort()' function
par(cex = 0.6)
barplot(sort(table(survey$Animal)), horiz = T, las = 1)  #still hard to read

We have many levels and lots of data cleaning is required.

What are the unique values?

In [ ]:
#What unique levels were assigned?
levels(survey$Animal)   #165 unique levels so far
length(levels(survey$Animal))

Things to clean:
*    Replace any instance of "A (animal)" with just the name of the animal
*    Deal with upper/lower case
*    Comments and punctuation(!)
*    Some weird final symbols
*    Maybe reduce number of classes or remove modifiers (I would be a dog so I can be fed and sleep all day.)
*    ????
*    
Before we go further, we create a new object that will be our working object (so we have the original object in case we mess up)

In [ ]:
animal <- survey$Animal
str(animal)

Use the `tolower()` function to make things lowercase.

In [ ]:
#Make everything lowercase - use the tolower() function
animal <- tolower(animal)

#How many levels now?
levels(animal)
str(animal)

**BAH!**   What happened?  The problem is that `survey$Animal` is a **FACTOR**.   By using the `tolower()` function, we turned it into a string (text) variable.

This is actually fine - we'll use this as a text variable and convert back to factor later if we desire.

SO - how many unique values now?

In [ ]:
unique(animal)     
length(unique(animal))  #down to 141

Let's sort alphabetically to make things easier to see.

In [ ]:
sort(unique(animal))

**NEXT**, remove any instance of "a (animal)" or "an (animal)" using the `gsub()` function.  `gsub()` looks for a text string and then replaces all instances with another text string inside of a character vector.

In [ ]:
#example of gsub
gsub("man", "fungus", c("Superman", "Batman", "manatee"))

As you can see, this might have unexpected consequences.

In [ ]:
gsub("a ", "", c("a dog", "a cat", "california condor"))

**PROBLEM** : we found all instances of `a ` (meaning the letter a followed by a space) and replaced them. This caused new problems : e.g.  `california condor` becames `californicondor`.

We want to replace ONLY when at the begining of the string.  Here's the solution : use `^` or `$`.

In [ ]:
# Use a ^ at the start of the search string to indicate a search only at the beginning of a string
sort(unique(gsub("^a ", "", animal)))

#Use a $ to indicate search only at the end of a string
#this code would delete the letter r at the end of a string.
#sort(unique(gsub("r$", "", animal)))

This uses what is called a **Regular Expression*

*   A regular expression is a special text string for describing a search pattern
*   Programming languages all have regular expressions
*   R has many recognized regular expressions
*   In this case `^` means "search only at the beginning of the string"

Here are some links to learn more about regular expressions.

[A link to learn about regular expressions in general](https://en.wikipedia.org/wiki/Regular_expression)

[A link to commonly used regular expresions in R](https://github.com/rstudio/cheatsheets/blob/main/regex.pdf)

**SO** - let's actually make a few replacements


In [ ]:
#Take out 'a ' at the beginning of words, write over the animal object
animal <- gsub("^a ", "", animal)

#Take out 'an ' at the beginning of words
animal <- gsub("^an ", "", animal)

#Fix instances of 'i would be a'
animal <- gsub("^i would be a ", "", animal)

#Fix instances of 'if i could be any animal i would be a '
animal <- gsub("^if i could be any animal i would be a ", "", animal)

#Now what are unique values?
sort(unique(animal)) 
length(sort(unique(animal)))  #down to 131

Next, we have some trailing spaces.   The '$' character denotes 'end of string' and we use an example of the escape character `\`

In [ ]:
#The code below seems like it should work.
animal <- gsub(" $","", animal)
#sort(unique(animal))

#Another idea - specify that we are looking for a 'space' character.
#Single \ stands for 'escape character', i.e. we're not talking about text, and \s stands for 'space character'.
animal <- gsub("\\s$", "", animal)
#sort(unique(animal))

#Let's try the trimws() function
trimws(" this is a test ")
animal <- trimws(animal, which = "both")
sort(unique(animal))

We still have trailing spaces at the end of words which are clearly some other **unusual characters**.  

Different platforms/devices use different characters to indicate the end of a line.  These characters may be hidden and LOOK like spaces, but they are not space characters.

In addition, we have a web address, periods, etc.

We now use `[]` square brackets to ONLY allow particular hexidecimal characters. We also use the [^...] to indicate **NOT**.  This is another regular expression.


In [ ]:
#The square brackets indicate that we're looking for particular characters.
#^ indicates we want anything that is NOT the identified characters.
#A quick example:
(tempvec <- c("This is test 9", "This is test nine", "this is test 9.", "9"))

grep("[0-9]", tempvec)
grep("[^0-9]", tempvec)
grep("[A-Z]", tempvec)
grep("[^A-Z]", tempvec)

In [ ]:
#first we turn them all in '. This is so we can see where the special characters are hiding!
animal <- gsub("[^0-9A-Za-z/' ]", "/" , animal)

#Pause for quick copilot discussion - ask what line of code above does

#check out cat
sort(unique(animal))

#Now, remove all of the /

animal <- gsub("/", "" , animal)
sort(unique(animal))
length(sort(unique(animal)))    #now 120

Let's deal with a few mis-spellings.

In [ ]:
#fix for humming bird
animal <- gsub("humming bird", "hummingbird", animal)

#fix spelling for axolotl
animal <- gsub("axolotyl", "axolotl", animal)

sort(unique(animal))
length(sort(unique(animal)))    #now 118

**NEXT** - deal with animals with modifiers (like 'dog in a rich suburban family or maybe a sloth')    Replace any text before or after animal name.

In [ ]:
#We could do one animal at a time
dogtrial <- gsub(".*dog.*", "dog", animal)
sort(unique(dogtrial))  

The regular expresion period `.` stands for 'any character'.   The regular expression `*` stands for 'any number of times'.

Of course, there are other animals besides dog that have modifiers.   We could modify the code above for each animal.  Or  . . . we could make a vector of animals to fix  **AND** let's use our first loop!

The code below also is our next use of the `paste()`function.   `paste()` is useful for putting multiple character values together into a single string.  Note that `sep = ""` specifies the character to put between parts of paste (i.e. don't put anything between bits. By default, `paste()` puts a single space between each bit)

In [ ]:
#An second example of the paste function.  Note it puts in a single space between each bit by default.
paste("Hi All,", round(3.14159, 2), "is my favorite", "number!")
paste0("Hi All,", round(3.14159, 2), "is my favorite", "number!")


In [ ]:
#Create a temporary character vector of animal names that appear with modifiers.  This code requires judgement choices.
#ALSO - NOTE ORDER MATTERS!!!!    FIX PANDA BEFORE BEAR.
anvec <- c("panda", "wolf","dog","koala", "human", "bear","lion", "parrot", "penguin", "eagle", "monkey", "dolphin", 
           "narwhal", "cat", "golden retriever", "tortoise", "whale", "sloth", "owl", "turtle", "hawk", "secretary bird", 
           "jaguar", "clarks nutcracker", "tiger", "falcon", "squirrel")

#These next two lines are just for testing as I'll show in class
#i <- 3
#rm(i)

#Loop our way through each animal on the list and remove words before and after.
for (i in 1:length(anvec)){
   animal <- gsub(paste0(".*", anvec[i] ,".*"), anvec[i], animal)
}
sort(unique(animal))   #down to 80

Notice that I decided panda should win over bear.  If I had put bear first in the list, it would have turned 'panda bear' into 'bear'.

Let's make a table and see about frequency


In [ ]:
table1 <- data.frame(sort(table(animal), decreasing = T))
table1
sort(unique(animal))  #still at 80


We're getting closer - but lots of singletons still.  I decided to do some grouping: these are just my preferences - you could make very different choices!

Make these changes.

In [ ]:
#I make regular of the | operator which stands for 'or'.

#create a backup in case we change our minds
animal_backup <- animal
#animal <- animal_backup

#NOTE - some of these replacements are from other years - but they work just the same since they only make replacements if found

#dog
animal <- gsub("australian shepherd|golden retriever|handsome dan|puppy|labrador|goldendoodle|golden pomeranian|puppy|chihuahua", "dog", animal)

#Birds
animal <- gsub("goshawk|falcon|owl|eagle|hawk|condor", "raptor", animal)
animal <- gsub("crow|cardinal|hummingbird|parrot|peacock|dove|clarks nutcracker|duck|loon|pidgeon|pigeon|secretary bird|swan|seagull", "bird", animal)

#Bunny
animal <- gsub("bunny", "rabbit", animal)

#Large cats
animal <- gsub("black panther|cheetah|jaguar|lion|liger|tiger|leopard|lynx|panther", "large cat", animal)

#large sea mammals
animal <- gsub("whale|orca|dolphin|porpoise|narwhal", "whale or dolphin", animal)

#primates
animal <- gsub("gorilla|monkey|gibbon|bonobo|orangutan|chimpanzee", "primates", animal)

#humans
animal <- gsub("jdrs hes a beast", "human", animal)

#Other water creatures with fewer than 3 counts
animal <- gsub("axolotl|giant squid|shark|jelly fish|mantis shrimp|seahorse|tardigrade", "other water creature", animal)

#Imaginary creatures
animal <- gsub("tulkun from avatar 2|puss in boots|judy from zootopia|pikachu|dragon", "imaginary", animal)

#remove missing values
animal <- animal[animal != ""]

#Check where we are
table1 <- data.frame(sort(table(animal),decreasing=T))
table1    #Down to 40 categories
sort(unique(animal))

**LAST STEP** - everything from 'chameleon' down we'll call `misc`.

In [ ]:
#chameleon is in row 21 of our temporary table
#rm(i)
for (i in 21:40){                      #start in row 21, finish in row 42
  animal <- gsub(paste0("^", as.character(table1[i, 1])), "misc. animal", animal)    
  #lookup table, find in animal, call misc.
}

#Check where we are
table2 <- data.frame(sort(table(animal), decreasing = T))
table2    #Down to 21 categories

In [ ]:
#Make table of results and store in finaltab
(finaltab <- table(animal))

**LAST thing** - We make a sorted barplot with added percentages on the labels

In [ ]:
#Calculate percents, round to nearest percent
percents <- round(finaltab/sum(finaltab)*100, 1)
percents

In [ ]:
names(finaltab) <- paste0(names(finaltab)," (", percents, "%)")

#Check where we're at
finaltab

par(mar = c(5, 12, 4, 2), cex = 0.6)
#FINAL PLOT
barplot(sort(finaltab), horiz = T, las = 1, col = "blue", main = "Animals Students Would Be", xlab = "Count") 

The End!